# 02 — Nettoyage et construction du graphe

Ce notebook démontre le pipeline complet de nettoyage et de création du graphe.

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, '../src')

from load_data import load_paysim
from clean_data import clean_data, save_cleaned
from build_graph import build_nodes, build_edges, build_networkx_graph, save_nodes_edges
from compute_features import compute_all_features, save_features

%matplotlib inline

In [ ]:
# Chargement
df_raw = load_paysim()
print(f'Données brutes : {len(df_raw):,} lignes')

In [ ]:
# Nettoyage
df_clean = clean_data(df_raw)
save_cleaned(df_clean)
print(f'Données nettoyées : {len(df_clean):,} lignes')
df_clean.head()

In [ ]:
# Construction du graphe
edges = build_edges(df_clean)
nodes = build_nodes(df_clean)
G = build_networkx_graph(nodes, edges)
save_nodes_edges(nodes, edges)

print(f'Nœuds : {G.number_of_nodes():,}')
print(f'Arêtes : {G.number_of_edges():,}')
print(f'Nœuds frauduleux : {nodes["is_fraud_node"].sum():,}')

In [ ]:
# Analyse de la structure du graphe
degrees = [d for n, d in G.degree()]
plt.figure(figsize=(10, 5))
plt.hist(degrees, bins=50, log=True, color='steelblue', edgecolor='black')
plt.title('Distribution des degrés (échelle log)')
plt.xlabel('Degré')
plt.ylabel('Fréquence (log)')
plt.show()

print(f'Degré moyen : {np.mean(degrees):.2f}')
print(f'Degré max : {max(degrees)}')

In [ ]:
# Calcul des features
features = compute_all_features(df_clean, nodes)
save_features(features)
print(f'Features shape : {features.shape}')
features.describe()

In [ ]:
# Visualisation des features importantes
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
cols = ['tx_sent_count', 'tx_recv_count', 'total_amount',
        'tx_sent_fraud_rate', 'tx_sent_unique_receivers', 'tx_sent_recv_ratio']

for ax, col in zip(axes.flatten(), cols):
    if col in features.columns:
        data = np.log1p(features[col])
        ax.hist(data, bins=40, color='teal', edgecolor='black', alpha=0.7)
        ax.set_title(f'log1p({col})')
        ax.set_xlabel('Valeur')
        ax.set_ylabel('Fréquence')

plt.tight_layout()
plt.show()

In [ ]:
# Visualisation d'un sous-graphe des nœuds frauduleux
fraud_nodes = nodes[nodes['is_fraud_node'] == 1]['node_id'].tolist()[:50]
subgraph = G.subgraph(fraud_nodes)

plt.figure(figsize=(12, 10))
pos = nx.spring_layout(subgraph, seed=42)
nx.draw_networkx(
    subgraph, pos,
    node_color='red', node_size=100,
    edge_color='gray', alpha=0.7,
    with_labels=False, arrows=True
)
plt.title('Sous-graphe des 50 premiers nœuds frauduleux')
plt.axis('off')
plt.show()